# Análise de Agrupamento (Clustering)

**Ciência de Dados — Aula 12**

**Dataset:** Medidas Corporais Sintéticas (Cintura + Comprimento da Perna)

**Objetivos:**
- Compreender clustering como aprendizado não supervisionado
- Aplicar e comparar três métodos: Hierárquico, K-Means e DBSCAN
- Interpretar resultados e escolher o método adequado

**Ferramentas:** Python, Plotly, scikit-learn, scipy

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import json, os

EXPORT_DIR = '../images/plotly/'

def export_plotly(fig, name):
    os.makedirs(EXPORT_DIR, exist_ok=True)
    fig_dict = fig.to_dict()
    with open(os.path.join(EXPORT_DIR, f'{name}.json'), 'w') as f:
        json.dump(fig_dict, f, default=str)
    print(f'Exported: {name}.json')

def dracula_layout(title='', **kwargs):
    return dict(
        title=title,
        paper_bgcolor='#282a36',
        plot_bgcolor='#282a36',
        font=dict(color='#f8f8f2', size=13),
        xaxis=dict(gridcolor='#44475a', zerolinecolor='#44475a', **kwargs.get('xaxis', {})),
        yaxis=dict(gridcolor='#44475a', zerolinecolor='#44475a', **kwargs.get('yaxis', {})),
        **{k: v for k, v in kwargs.items() if k not in ('xaxis', 'yaxis')}
    )

DRACULA_COLORS = ['#8be9fd', '#ff79c6', '#50fa7b', '#f1fa8c', '#bd93f9', '#ffb86c', '#ff5555', '#6272a4']

np.random.seed(42)
print('Setup completo!')

## O Problema das Numerações de Calças

Por que numerações de calças existem? Por que duas pessoas que usam tamanho 42 podem ter corpos muito diferentes?

Fabricantes precisam **comprimir a variação contínua do corpo humano** em um número limitado de categorias.

> **Isso é essencialmente um problema de clustering!**

### Variáveis que vamos usar:
- **Cintura (cm):** circunferência da cintura
- **Perna (cm):** comprimento da perna / gancho

Duas pessoas podem ter a mesma cintura mas pernas muito diferentes. Como agrupá-las?

In [ ]:
n_per_group = 60

group_a = pd.DataFrame({
    'cintura': np.random.normal(72, 5, n_per_group),
    'perna': np.random.normal(72, 4, n_per_group),
    'grupo_real': 'A — Baixo/Magro'
})
group_b = pd.DataFrame({
    'cintura': np.random.normal(74, 5, n_per_group),
    'perna': np.random.normal(92, 4, n_per_group),
    'grupo_real': 'B — Alto/Magro'
})
group_c = pd.DataFrame({
    'cintura': np.random.normal(98, 5, n_per_group),
    'perna': np.random.normal(74, 4, n_per_group),
    'grupo_real': 'C — Baixo/Largo'
})
group_d = pd.DataFrame({
    'cintura': np.random.normal(102, 5, n_per_group),
    'perna': np.random.normal(94, 4, n_per_group),
    'grupo_real': 'D — Alto/Largo'
})

noise = pd.DataFrame({
    'cintura': np.random.uniform(65, 115, 15),
    'perna': np.random.uniform(65, 100, 15),
    'grupo_real': 'Ruído'
})

outliers = pd.DataFrame({
    'cintura': [60, 115, 85, 90, 70],
    'perna': [95, 65, 105, 60, 100],
    'grupo_real': 'Outlier'
})

dados = pd.concat([group_a, group_b, group_c, group_d, noise, outliers], ignore_index=True)
dados = dados.sample(frac=1, random_state=42).reset_index(drop=True)

print(f'Total de pontos: {len(dados)}')
print(f'\nDistribuição dos grupos reais:')
print(dados['grupo_real'].value_counts())
display(dados.head(10))

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=dados['cintura'],
    y=dados['perna'],
    mode='markers',
    marker=dict(color='#6272a4', size=8, opacity=0.6),
    text=[f'Cintura: {c:.0f}, Perna: {p:.0f}' for c, p in zip(dados['cintura'], dados['perna'])],
    hovertemplate='%{text}<extra></extra>',
    showlegend=False
))

fig.update_layout(**dracula_layout('Medidas Corporais — Dados Brutos',
                                    xaxis=dict(title='Cintura (cm)'),
                                    yaxis=dict(title='Comprimento da Perna (cm)'),
                                    height=600))

export_plotly(fig, 'plot_raw_data')
fig.show()

### Discussão

Observe o gráfico acima. Responda mentalmente:
- Você consegue identificar grupos visualmente?
- Quantos grupos você vê?
- As fronteiras entre grupos são claras?
- Existem pontos que parecem não pertencer a nenhum grupo?

> Lembre-se: no mundo real, **não temos rótulos**. O algoritmo precisa descobrir os grupos sozinho!

## 1. Distância e Similaridade

Vocês já usaram **distância Euclidiana** no KNN para classificar pontos. A mesma ideia é a base do clustering!

**Se dois pontos estão próximos no espaço de features, eles são similares.**

$$d(p, q) = \sqrt{(p_1 - q_1)^2 + (p_2 - q_2)^2}$$

A diferença: no KNN, usamos distância para encontrar vizinhos com rótulos conhecidos. No clustering, usamos distância para **agrupar pontos sem rótulos**.

In [ ]:
p_idx, q_idx = 0, 1
p_point = dados.iloc[p_idx][['cintura', 'perna']].values
q_point = dados.iloc[q_idx][['cintura', 'perna']].values
dist_eucl = np.linalg.norm(p_point - q_point)

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=dados['cintura'], y=dados['perna'],
    mode='markers', marker=dict(color='#6272a4', size=8, opacity=0.4),
    showlegend=False, hoverinfo='skip'
))

fig.add_trace(go.Scatter(
    x=[p_point[0]], y=[p_point[1]],
    mode='markers', marker=dict(color='#8be9fd', size=14, line=dict(width=2, color='#f8f8f2')),
    name=f'Ponto P (índice {p_idx})',
    hovertemplate=f'P: ({p_point[0]:.1f}, {p_point[1]:.1f})<extra></extra>'
))

fig.add_trace(go.Scatter(
    x=[q_point[0]], y=[q_point[1]],
    mode='markers', marker=dict(color='#ff79c6', size=14, line=dict(width=2, color='#f8f8f2')),
    name=f'Ponto Q (índice {q_idx})',
    hovertemplate=f'Q: ({q_point[0]:.1f}, {q_point[1]:.1f})<extra></extra>'
))

fig.add_trace(go.Scatter(
    x=[p_point[0], q_point[0]], y=[p_point[1], q_point[1]],
    mode='lines', line=dict(color='#f1fa8c', width=2, dash='dash'),
    showlegend=False, hoverinfo='skip'
))

mid_x = (p_point[0] + q_point[0]) / 2
mid_y = (p_point[1] + q_point[1]) / 2
fig.add_annotation(
    x=mid_x, y=mid_y,
    text=f'd = {dist_eucl:.2f} cm',
    showarrow=False,
    font=dict(color='#f1fa8c', size=14),
    bgcolor='#282a36', bordercolor='#f1fa8c', borderwidth=1
)

fig.update_layout(**dracula_layout('Distância Euclidiana entre Dois Pontos',
                                    xaxis=dict(title='Cintura (cm)'),
                                    yaxis=dict(title='Comprimento da Perna (cm)'),
                                    height=600))

export_plotly(fig, 'plot_distance_demo')
fig.show()

In [ ]:
from scipy.spatial.distance import pdist, squareform

sample_idx = dados.sample(25, random_state=42).index
dados_sample = dados.loc[sample_idx, ['cintura', 'perna']].values
dist_matrix = squareform(pdist(dados_sample, metric='euclidean'))

fig = go.Figure(go.Heatmap(
    z=dist_matrix,
    colorscale=[
        [0, '#282a36'],
        [0.25, '#50fa7b'],
        [0.5, '#f1fa8c'],
        [0.75, '#ffb86c'],
        [1.0, '#ff5555']
    ],
    hovertemplate='Amostra %{x} → Amostra %{y}<br>Dist: %{z:.1f}<extra></extra>',
    showscale=True,
    colorbar=dict(title=dict(text='Distância', font=dict(color='#f8f8f2')), tickfont=dict(color='#f8f8f2'))
))

fig.update_layout(**dracula_layout('Matriz de Distância (25 amostras)',
                                    xaxis=dict(title='Amostra', showticklabels=False),
                                    yaxis=dict(title='Amostra', showticklabels=False),
                                    height=600, width=650))

export_plotly(fig, 'plot_distance_matrix')
fig.show()

## 2. Clustering Hierárquico

### Ideia Principal
Cada ponto começa sozinho. Depois, os pontos mais próximos são **fundidos** progressivamente até formar uma única árvore.

### Abordagem Aglomerativa:
1. Cada ponto é um cluster
2. Calcule a distância entre todos os pares de clusters
3. **Funda os dois clusters mais próximos**
4. Repita até ter um único cluster

O resultado é um **dendrograma** — uma árvore que mostra TODAS as fusões.

> **Vantagem:** Não precisamos definir K antecipadamente — podemos "cortar" a árvore na altura que fizer sentido.

In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import pdist, squareform

X = dados[['cintura', 'perna']].values

Z_ward = linkage(X, method='ward')

print(f'Primeiras 10 fusões (linkage Ward):')
print(f'idx1  idx2  distância  novos_pontos')
print('-' * 45)
for i in range(10):
    print(f'{int(Z_ward[i,0]):4d}  {int(Z_ward[i,1]):4d}  {Z_ward[i,2]:9.2f}  {int(Z_ward[i,3]):4d}')

In [ ]:
from scipy.cluster.hierarchy import dendrogram

plt_result = dendrogram(Z_ward, no_plot=True, no_labels=True)
icoord = np.array(plt_result['icoord'])
dcoord = np.array(plt_result['dcoord'])

fig = go.Figure()
for i in range(len(icoord)):
    fig.add_trace(go.Scatter(
        x=icoord[i], y=dcoord[i],
        mode='lines', line=dict(color='#6272a4', width=1),
        showlegend=False, hoverinfo='skip'
    ))

fig.update_layout(**dracula_layout('Dendrograma — Clustering Hierárquico (Ward)',
                                    xaxis=dict(title='Amostras', showticklabels=False),
                                    yaxis=dict(title='Distância de Fusão'),
                                    height=600, width=1000))

cut_height = Z_ward[-3, 2]
fig.add_hline(y=cut_height, line_dash='dash', line_color='#ff5555',
              annotation_text=f'Corte: K=4', annotation_position='top right')

export_plotly(fig, 'plot_dendrogram')
fig.show()

### Como Ler um Dendrograma

- **Eixo Y (vertical):** distância em que dois clusters foram fundidos
- **Fusões baixas:** clusters muito similares (fundidos cedo)
- **Fusões altas:** clusters muito diferentes (fundidos tarde)
- **Saltos grandes:** indicam separações naturais entre grupos

A **linha vermelha tracejada** mostra onde "cortamos" a árvore para obter 4 clusters.

> Quanto mais alto o salto antes do corte, mais distintos são os grupos!

In [ ]:
from scipy.cluster.hierarchy import fcluster

labels_hier = fcluster(Z_ward, t=4, criterion='maxclust')
dados['cluster_hier'] = labels_hier

fig = go.Figure()
for i in sorted(dados['cluster_hier'].unique()):
    mask = dados['cluster_hier'] == i
    fig.add_trace(go.Scatter(
        x=dados.loc[mask, 'cintura'],
        y=dados.loc[mask, 'perna'],
        mode='markers',
        marker=dict(color=DRACULA_COLORS[i-1], size=8, opacity=0.7),
        name=f'Cluster {i}',
        text=[f'Cintura: {c:.0f}, Perna: {p:.0f}' for c, p in zip(dados.loc[mask, 'cintura'], dados.loc[mask, 'perna'])],
        hovertemplate='%{text}<extra></extra>'
    ))

fig.update_layout(**dracula_layout('Clustering Hierárquico (Ward, K=4)',
                                    xaxis=dict(title='Cintura (cm)'),
                                    yaxis=dict(title='Comprimento da Perna (cm)'),
                                    height=600))

export_plotly(fig, 'plot_hierarchical_clusters')
fig.show()

In [ ]:
from scipy.cluster.hierarchy import linkage, fcluster

X = dados[['cintura', 'perna']].values
linkage_methods = ['single', 'complete', 'average', 'ward']

fig = make_subplots(rows=2, cols=2,
                     subplot_titles=['Single Linkage', 'Complete Linkage', 'Average Linkage', 'Ward Linkage'])

for idx, method in enumerate(linkage_methods):
    Z = linkage(X, method=method)
    labels = fcluster(Z, t=4, criterion='maxclust')
    row, col = idx // 2 + 1, idx % 2 + 1

    for k in sorted(set(labels)):
        mask = labels == k
        fig.add_trace(go.Scatter(
            x=X[mask, 0], y=X[mask, 1],
            mode='markers',
            marker=dict(color=DRACULA_COLORS[k-1], size=6, opacity=0.7),
            showlegend=False,
            name=f'Cluster {k}',
            hoverinfo='skip'
        ), row=row, col=col)

fig.update_layout(**dracula_layout('Comparação de Métodos de Linkage',
                                    height=700, width=900,
                                    showlegend=False))

export_plotly(fig, 'plot_linkage_comparison')
fig.show()

### Clustering Hierárquico — Prós e Contras

| Aspecto | Detalhe |
|---------|---------|
| ✅ Interpretabilidade | Dendrograma é intuitivo e visual |
| ✅ Sem K inicial | Podemos escolher o corte depois |
| ✅ Exploratório | Mostra a estrutura completa dos dados |
| ❌ Custo computacional | O(n²) a O(n³) — não escala bem |
| ❌ Sensível ao linkage | Diferentes métodos → diferentes resultados |
| ❌ Sem reatribuição | Uma vez fundido, não pode desfazer |

> **Quando usar:** Dados pequenos/médios, quando a visualização da estrutura hierárquica é importante.

---

## 📋 Exercício 1 — Linkage e Interpretação

**Objetivo:** Explorar como o método de linkage afeta os clusters.

1. Altere o método de linkage no código acima e observe como os clusters mudam
2. Qual método parece mais adequado para os nossos dados de medidas corporais?
3. Experimente cortar o dendrograma com K=3 e K=5. O que muda?

Escreva suas conclusões na célula abaixo.

In [ ]:
# Seu código aqui

## 3. K-Means: Centroides como Moldes

### A Analogia do Fabricante

Imagine que uma empresa decide: **"Vamos criar 4 tipos representativos de corpo."**

Cada **centroide** vira um **molde** — o padrão que representa aquele grupo.

### O Algoritmo (4 passos):
1. **Inicializar** K centroides aleatoriamente
2. **Atribuir** cada ponto ao centroide mais próximo
3. **Recalcular** cada centroide como a média dos pontos atribuídos
4. **Repetir** passos 2-3 até convergir (centroides não mudam)

> **Diferença do Hierárquico:** K-Means precisa que você defina K **antes** de rodar. Mas é muito mais rápido!

In [ ]:
K = 4
X = dados[['cintura', 'perna']].values

centroids = np.array([[70, 90], [100, 85], [80, 70], [95, 95]])

steps = []
for iteration in range(4):
    distances = np.array([[np.linalg.norm(x - c) for c in centroids] for x in X])
    labels = np.argmin(distances, axis=1)

    steps.append({'centroids': centroids.copy(), 'labels': labels.copy()})

    new_centroids = np.array([X[labels == k].mean(axis=0) if (labels == k).any() else centroids[k] for k in range(K)])
    centroids = new_centroids

titles = ['Inicialização', 'Iteração 1', 'Iteração 2', 'Convergido']
fig = make_subplots(rows=2, cols=2, subplot_titles=titles)

for idx, (step, title) in enumerate(zip(steps, titles)):
    row, col = idx // 2 + 1, idx % 2 + 1
    for k in range(K):
        mask = step['labels'] == k
        fig.add_trace(go.Scatter(
            x=X[mask, 0], y=X[mask, 1],
            mode='markers', marker=dict(color=DRACULA_COLORS[k], size=6, opacity=0.6),
            showlegend=(idx == 0), name=f'Grupo {k}',
            hoverinfo='skip'
        ), row=row, col=col)

    fig.add_trace(go.Scatter(
        x=step['centroids'][:, 0], y=step['centroids'][:, 1],
        mode='markers', marker=dict(color='#ff5555', size=16, symbol='x', line=dict(width=3)),
        showlegend=False, hoverinfo='skip'
    ), row=row, col=col)

fig.update_layout(**dracula_layout('K-Means — Evolução das Iterações',
                                    height=700, width=900,
                                    showlegend=True))

export_plotly(fig, 'plot_kmeans_steps')
fig.show()

In [ ]:
from sklearn.cluster import KMeans

K = 4
kmeans = KMeans(n_clusters=K, random_state=42, n_init=10)
labels_km = kmeans.fit_predict(X)
dados['cluster_kmeans'] = labels_km

fig = go.Figure()
for k in range(K):
    mask = dados['cluster_kmeans'] == k
    fig.add_trace(go.Scatter(
        x=dados.loc[mask, 'cintura'],
        y=dados.loc[mask, 'perna'],
        mode='markers',
        marker=dict(color=DRACULA_COLORS[k], size=8, opacity=0.7),
        name=f'Cluster {k}',
    ))

fig.add_trace(go.Scatter(
    x=kmeans.cluster_centers_[:, 0],
    y=kmeans.cluster_centers_[:, 1],
    mode='markers',
    marker=dict(color='#ff5555', size=18, symbol='x', line=dict(width=3)),
    name='Centroides',
    hovertemplate='Centroide (%{x:.1f}, %{y:.1f})<extra></extra>'
))

fig.update_layout(**dracula_layout('K-Means (K=4) — Resultado Final',
                                    xaxis=dict(title='Cintura (cm)'),
                                    yaxis=dict(title='Comprimento da Perna (cm)'),
                                    height=600))

export_plotly(fig, 'plot_kmeans_result')
fig.show()

print(f'Inércia (WCSS): {kmeans.inertia_:.2f}')
print(f'Centroides:')
for i, c in enumerate(kmeans.cluster_centers_):
    print(f'  Cluster {i}: cintura={c[0]:.1f} cm, perna={c[1]:.1f} cm')

In [ ]:
from sklearn.metrics import silhouette_score

max_k = 12
inertias = []
silhouettes = []

for k in range(1, max_k + 1):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X)
    inertias.append(km.inertia_)
    if k > 1:
        silhouettes.append(silhouette_score(X, km.labels_))
    else:
        silhouettes.append(0)

fig = make_subplots(rows=1, cols=2, subplot_titles=['Método do Cotovelo', 'Silhouette Score'])

fig.add_trace(go.Scatter(
    x=list(range(1, max_k + 1)), y=inertias,
    mode='lines+markers', marker=dict(color='#50fa7b', size=10),
    line=dict(color='#50fa7b', width=2.5),
    name='Inércia'
), row=1, col=1)

fig.add_vline(x=4, line_dash='dash', line_color='#ff5555',
              annotation_text='K=4', row=1, col=1)

fig.add_trace(go.Scatter(
    x=list(range(2, max_k + 1)), y=silhouettes[1:],
    mode='lines+markers', marker=dict(color='#bd93f9', size=10),
    line=dict(color='#bd93f9', width=2.5),
    name='Silhouette'
), row=1, col=2)

fig.add_vline(x=4, line_dash='dash', line_color='#ff5555',
              annotation_text='K=4', row=1, col=2)

fig.update_layout(**dracula_layout('Escolhendo o Número de Clusters',
                                    height=500, width=1000,
                                    showlegend=False))
fig.update_xaxes(title_text='K (número de clusters)', row=1, col=1)
fig.update_xaxes(title_text='K (número de clusters)', row=1, col=2)
fig.update_yaxes(title_text='Inércia (WCSS)', row=1, col=1)
fig.update_yaxes(title_text='Silhouette Score', row=1, col=2)

export_plotly(fig, 'plot_elbow')
fig.show()

### K-Means — Prós e Contras

| Aspecto | Detalhe |
|---------|---------|
| ✅ Rápido | O(n·K·i) — escala para milhões de pontos |
| ✅ Intuitivo | Centroides como "moldes representativos" |
| ✅ Simples | Poucos parâmetros (apenas K) |
| ❌ Precisa de K | Precisamos definir o número de clusters antes |
| ❌ Esférico | Assume clusters com forma circular/esférica |
| ❌ Sensível à inicialização | Diferentes seeds → diferentes resultados |
| ❌ Não trata ruído | Todos os pontos são forçados em um cluster |

> **Quando usar:** Dados grandes, clusters aproximadamente esféricos, quando já temos ideia de K.

Mas o que acontece quando os clusters **não são esféricos**?

---

## 📋 Exercício 2 — Encontrando o K Ideal

**Objetivo:** Usar o método do cotovelo e silhouette para determinar K.

1. Analise os gráficos acima. O cotovelo confirma K=4?
2. Qual valor de K tem o maior silhouette score?
3. O que acontece com K=3? E com K=5? Execute K-Means para esses valores e compare visualmente.

Use a célula abaixo para suas experimentações.

In [ ]:
# Seu código aqui

## 4. Quando K-Means Falha

K-Means assume que os clusters são **esféricos** e de **tamanho similar**.

Mas e se os dados tiverem:
- Clusters com **formas arbitrárias** (meia-lua, espirais)
- **Densidades diferentes** (grupos densos e esparsos misturados)
- Muito **ruído** (pontos isolados)

Vamos ver o que acontece...

In [ ]:
from sklearn.datasets import make_moons, make_circles

X_moons, y_moons = make_moons(n_samples=300, noise=0.08, random_state=42)
X_circles, y_circles = make_circles(n_samples=300, noise=0.05, factor=0.5, random_state=42)

km_moons = KMeans(n_clusters=2, random_state=42, n_init=10).fit(X_moons)
km_circles = KMeans(n_clusters=2, random_state=42, n_init=10).fit(X_circles)

fig = make_subplots(rows=1, cols=2,
                     subplot_titles=['Meia-Lua (K-Means K=2)', 'Círculos (K-Means K=2)'])

for k in range(2):
    mask = km_moons.labels_ == k
    fig.add_trace(go.Scatter(
        x=X_moons[mask, 0], y=X_moons[mask, 1],
        mode='markers', marker=dict(color=DRACULA_COLORS[k], size=7, opacity=0.7),
        showlegend=False, hoverinfo='skip'
    ), row=1, col=1)

for k in range(2):
    mask = km_circles.labels_ == k
    fig.add_trace(go.Scatter(
        x=X_circles[mask, 0], y=X_circles[mask, 1],
        mode='markers', marker=dict(color=DRACULA_COLORS[k], size=7, opacity=0.7),
        showlegend=False, hoverinfo='skip'
    ), row=1, col=2)

fig.update_layout(**dracula_layout('K-Means Falha em Formas Não-Esféricas',
                                    height=500, width=1000,
                                    showlegend=False))

export_plotly(fig, 'plot_kmeans_fails')
fig.show()

## 5. DBSCAN: Agrupamento por Densidade

### Ideia Central
DBSCAN pergunta: **"Esta região é densa o suficiente para ser um cluster?"**

Diferente do K-Means que pergunta "Qual centroide é o mais próximo?"

### Conceitos Fundamentais:
- **ε (epsilon):** raio de vizinhança
- **min_samples:** número mínimo de pontos na vizinhança
- **Core point:** tem ≥ min_samples pontos dentro de ε
- **Border point:** dentro de ε de um core point, mas não tem min_samples vizinhos
- **Noise point:** nem core nem border → **ruído/outlier**

> **Vantagem:** DBSCAN pode identificar clusters de qualquer forma e detectar outliers automaticamente!

In [ ]:
eps_demo = 0.2
min_samples_demo = 5

core_idx = 150
neighbors = np.where(np.linalg.norm(X_moons - X_moons[core_idx], axis=1) <= eps_demo)[0]

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=X_moons[:, 0], y=X_moons[:, 1],
    mode='markers', marker=dict(color='#6272a4', size=7, opacity=0.4),
    showlegend=False, hoverinfo='skip'
))

fig.add_trace(go.Scatter(
    x=X_moons[neighbors, 0], y=X_moons[neighbors, 1],
    mode='markers', marker=dict(color='#50fa7b', size=9, opacity=0.8),
    name=f'Vizinhos ({len(neighbors)} pontos)',
    hoverinfo='skip'
))

fig.add_trace(go.Scatter(
    x=[X_moons[core_idx, 0]], y=[X_moons[core_idx, 1]],
    mode='markers', marker=dict(color='#ff5555', size=14, line=dict(width=2, color='#f8f8f2')),
    name='Core Point',
    hoverinfo='skip'
))

fig.add_shape(type='circle',
              xref='x', yref='y',
              x0=X_moons[core_idx, 0] - eps_demo,
              y0=X_moons[core_idx, 1] - eps_demo,
              x1=X_moons[core_idx, 0] + eps_demo,
              y1=X_moons[core_idx, 1] + eps_demo,
              line=dict(color='#8be9fd', width=2, dash='dash'),
              fillcolor='rgba(139, 233, 253, 0.08)')

fig.add_annotation(
    x=X_moons[core_idx, 0], y=X_moons[core_idx, 1] + eps_demo + 0.05,
    text=f'ε = {eps_demo}<br>Vizinhos: {len(neighbors)} (min={min_samples_demo})',
    showarrow=False, font=dict(color='#8be9fd', size=13),
    bgcolor='#282a36', bordercolor='#8be9fd', borderwidth=1
)

fig.update_layout(**dracula_layout('DBSCAN — Vizinhança ε (epsilon)',
                                    xaxis=dict(title='X₁', scaleanchor='y', scaleratio=1),
                                    yaxis=dict(title='X₂'),
                                    height=600,
                                    showlegend=True))

export_plotly(fig, 'plot_epsilon_demo')
fig.show()

In [ ]:
from sklearn.cluster import DBSCAN

db_moons = DBSCAN(eps=0.2, min_samples=5).fit(X_moons)
n_clusters = len(set(db_moons.labels_)) - (1 if -1 in db_moons.labels_ else 0)
n_noise = (db_moons.labels_ == -1).sum()

fig = go.Figure()

for label in sorted(set(db_moons.labels_)):
    mask = db_moons.labels_ == label
    if label == -1:
        fig.add_trace(go.Scatter(
            x=X_moons[mask, 0], y=X_moons[mask, 1],
            mode='markers', marker=dict(color='#ff5555', size=10, symbol='x', opacity=0.8),
            name=f'Ruído ({mask.sum()} pontos)',
            hoverinfo='skip'
        ))
    else:
        fig.add_trace(go.Scatter(
            x=X_moons[mask, 0], y=X_moons[mask, 1],
            mode='markers', marker=dict(color=DRACULA_COLORS[label], size=8, opacity=0.7),
            name=f'Cluster {label} ({mask.sum()} pontos)',
            hoverinfo='skip'
        ))

fig.update_layout(**dracula_layout(f'DBSCAN nas Meias-Luas — {n_clusters} clusters, {n_noise} ruído',
                                    xaxis=dict(title='X₁'),
                                    yaxis=dict(title='X₂'),
                                    height=600))

export_plotly(fig, 'plot_dbscan_moons')
fig.show()

In [ ]:
from sklearn.cluster import DBSCAN

eps_values = [0.1, 0.2, 0.3]
min_samples_values = [5, 10]

fig = make_subplots(
    rows=2, cols=3,
    subplot_titles=[f'ε={e}, min_samples={m}' for m in min_samples_values for e in eps_values]
)

for row_idx, min_s in enumerate(min_samples_values):
    for col_idx, eps_val in enumerate(eps_values):
        db = DBSCAN(eps=eps_val, min_samples=min_s).fit(X_moons)
        n_cl = len(set(db.labels_)) - (1 if -1 in db.labels_ else 0)

        for label in sorted(set(db.labels_)):
            mask = db.labels_ == label
            if label == -1:
                fig.add_trace(go.Scatter(
                    x=X_moons[mask, 0], y=X_moons[mask, 1],
                    mode='markers', marker=dict(color='#ff5555', size=6, symbol='x', opacity=0.7),
                    showlegend=False, hoverinfo='skip'
                ), row=row_idx + 1, col=col_idx + 1)
            else:
                color_idx = label % len(DRACULA_COLORS)
                fig.add_trace(go.Scatter(
                    x=X_moons[mask, 0], y=X_moons[mask, 1],
                    mode='markers', marker=dict(color=DRACULA_COLORS[color_idx], size=6, opacity=0.7),
                    showlegend=False, hoverinfo='skip'
                ), row=row_idx + 1, col=col_idx + 1)

fig.update_layout(**dracula_layout('DBSCAN — Variação de Parâmetros (Meias-Luas)',
                                    height=700, width=1000,
                                    showlegend=False))

export_plotly(fig, 'plot_dbscan_params')
fig.show()

In [ ]:
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler

X_pants = dados[['cintura', 'perna']].values
X_scaled = StandardScaler().fit_transform(X_pants)

db_pants = DBSCAN(eps=0.5, min_samples=5).fit(X_scaled)
dados['cluster_dbscan'] = db_pants.labels_

n_clusters_db = len(set(db_pants.labels_)) - (1 if -1 in db_pants.labels_ else 0)
n_noise_db = (db_pants.labels_ == -1).sum()

fig = go.Figure()

for label in sorted(set(db_pants.labels_)):
    mask = db_pants.labels_ == label
    if label == -1:
        fig.add_trace(go.Scatter(
            x=dados.loc[mask, 'cintura'], y=dados.loc[mask, 'perna'],
            mode='markers', marker=dict(color='#ff5555', size=10, symbol='x', opacity=0.8),
            name=f'Ruído ({mask.sum()} pts)',
            hoverinfo='skip'
        ))
    else:
        color_idx = label % len(DRACULA_COLORS)
        fig.add_trace(go.Scatter(
            x=dados.loc[mask, 'cintura'], y=dados.loc[mask, 'perna'],
            mode='markers', marker=dict(color=DRACULA_COLORS[color_idx], size=8, opacity=0.7),
            name=f'Cluster {label} ({mask.sum()} pts)',
            hoverinfo='skip'
        ))

fig.update_layout(**dracula_layout(f'DBSCAN nas Medidas Corporais — {n_clusters_db} clusters, {n_noise_db} ruído',
                                    xaxis=dict(title='Cintura (cm)'),
                                    yaxis=dict(title='Comprimento da Perna (cm)'),
                                    height=600))

export_plotly(fig, 'plot_dbscan_pants')
fig.show()

### DBSCAN — Prós e Contras

| Aspecto | Detalhe |
|---------|---------|
| ✅ Sem K | Não precisa definir número de clusters |
| ✅ Formas arbitrárias | Detecta clusters de qualquer formato |
| ✅ Detecta ruído | Identifica outliers automaticamente |
| ❌ Sensível a ε e min_samples | Dificuldade em escolher parâmetros |
| ❌ Densidades variáveis | Problemas com clusters de densidades diferentes |
| ❌ Alta dimensionalidade | Distância perde significado em muitas dimensões |

> **Quando usar:** Dados com ruído, clusters não-esféricos, quando não sabemos K.

---

## 📋 Exercício 3 — DBSCAN na Prática

**Objetivo:** Ajustar os parâmetros do DBSCAN.

1. Aplique DBSCAN nos dados de medidas corporais variando `eps` de 0.3 a 1.0
2. Para cada valor, registre: número de clusters e pontos de ruído
3. Qual combinação de `eps` e `min_samples` produz o resultado mais interpretável?
4. Os pontos de ruído fazem sentido? Correspondem aos outliers que geramos?

Use a célula abaixo para suas experimentações.

In [ ]:
# Seu código aqui

## 6. Comparação Final

Vamos aplicar os três métodos no mesmo dataset e comparar lado a lado.

In [ ]:
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler

X = dados[['cintura', 'perna']].values
X_scaled = StandardScaler().fit_transform(X)

db_final = DBSCAN(eps=0.5, min_samples=5).fit(X_scaled)

fig = make_subplots(rows=1, cols=3,
                     subplot_titles=['Hierárquico (Ward)', 'K-Means (K=4)', 'DBSCAN'])

for i in sorted(dados['cluster_hier'].unique()):
    mask = dados['cluster_hier'] == i
    fig.add_trace(go.Scatter(
        x=dados.loc[mask, 'cintura'], y=dados.loc[mask, 'perna'],
        mode='markers', marker=dict(color=DRACULA_COLORS[i-1], size=6, opacity=0.7),
        showlegend=False, hoverinfo='skip'
    ), row=1, col=1)

for k in range(4):
    mask = dados['cluster_kmeans'] == k
    fig.add_trace(go.Scatter(
        x=dados.loc[mask, 'cintura'], y=dados.loc[mask, 'perna'],
        mode='markers', marker=dict(color=DRACULA_COLORS[k], size=6, opacity=0.7),
        showlegend=False, hoverinfo='skip'
    ), row=1, col=2)

for label in sorted(set(db_final.labels_)):
    mask = db_final.labels_ == label
    if label == -1:
        fig.add_trace(go.Scatter(
            x=dados.loc[mask, 'cintura'], y=dados.loc[mask, 'perna'],
            mode='markers', marker=dict(color='#ff5555', size=8, symbol='x', opacity=0.8),
            showlegend=False, hoverinfo='skip'
        ), row=1, col=3)
    else:
        color_idx = label % len(DRACULA_COLORS)
        fig.add_trace(go.Scatter(
            x=dados.loc[mask, 'cintura'], y=dados.loc[mask, 'perna'],
            mode='markers', marker=dict(color=DRACULA_COLORS[color_idx], size=6, opacity=0.7),
            showlegend=False, hoverinfo='skip'
        ), row=1, col=3)

fig.update_layout(**dracula_layout('Comparação: Hierárquico vs K-Means vs DBSCAN',
                                    height=500, width=1200,
                                    showlegend=False))

export_plotly(fig, 'plot_comparison')
fig.show()

### Tabela Comparativa

| Critério | Hierárquico | K-Means | DBSCAN |
|----------|:-----------:|:-------:|:------:|
| Precisa definir K? | Não (corte depois) | Sim | Não |
| Forma dos clusters | Qualquer | Esférico | Qualquer |
| Detecta ruído? | Não | Não | Sim |
| Escalabilidade | Baixa | Alta | Média |
| Parâmetros principais | linkage, K de corte | K | ε, min_samples |
| Interpretabilidade | Alta (dendrograma) | Média | Média |

### Reflexão Final

> **Clustering não tem verdade absoluta.** Dois fabricantes podem definir numerações diferentes, e ambos podem estar "certos". A escolha do algoritmo e dos parâmetros é uma **decisão de modelagem** que depende do contexto e dos objetivos.

---

## 📋 Exercício 4 — Projeto Final

**Objetivo:** Aplicar clustering em um dataset real.

**Dataset sugerido:** [Mall Customer Segmentation Data](https://www.kaggle.com/vjchoudhary7/customer-segmentation-tutorial-in-python)

Ou use o `sklearn.datasets.load_iris()` (ignorando os rótulos!)

### Instruções:
1. Carregue o dataset e explore as variáveis
2. Escolha 2 variáveis para visualização inicial
3. Aplique os três métodos: Hierárquico, K-Means e DBSCAN
4. Para K-Means: use o método do cotovelo para encontrar K
5. Para DBSCAN: experimente diferentes valores de ε e min_samples
6. Compare os resultados visualmente
7. **Qual método você recomendaria? Justifique com base nas características dos dados**

Use as células abaixo para desenvolver sua análise.

In [ ]:
# Seu código aqui

In [ ]:
# Seu código aqui

In [ ]:
# Seu código aqui

In [ ]:
# Seu código aqui